# Lab 3: Inheritance, Polymorphism, and Composition
## IST1012 — Python Programming II
**Difficulty: Above Medium** | **Estimated Time: 60 minutes**

This lab covers **inheritance** (creating specialized classes from
general ones), **polymorphism** (same interface, different behavior),
and **composition** (building complex objects from simpler ones).
These patterns are essential for designing extensible security tools.

## Learning Objectives
1. Create child classes that inherit from a parent class
2. Override methods to provide specialized behavior
3. Use `super()` to extend parent functionality
4. Distinguish inheritance ("is-a") from composition ("has-a")
5. Apply polymorphism to process different object types uniformly

---
## Part 1: Warm-Up (5-10 min)



**Q1.** Run the code below. What relationship does `list` have
with `object` in Python? What does `isinstance` tell us?

In [1]:
my_list = [1, 2, 3]
print(isinstance(my_list, list))
print(isinstance(my_list, object))
print(issubclass(list, object))

True
True
True


---
## Part 2: Core Concepts (10-15 min)

### Inheritance Basics

A **child class** inherits attributes and methods from a **parent class**
and can add or override them.

In [2]:
class SecurityEvent:
    def __init__(self, timestamp, severity):
        self.timestamp = timestamp
        self.severity = severity

    def describe(self):
        return "[" + self.timestamp + "] Severity: " + str(self.severity)

class MalwareEvent(SecurityEvent):
    def __init__(self, timestamp, severity, malware_name):
        SecurityEvent.__init__(self, timestamp, severity)
        self.malware_name = malware_name

    def describe(self):
        base = SecurityEvent.describe(self)
        return base + " | Malware: " + self.malware_name

event = MalwareEvent("2025-03-23 14:00", 8, "Trojan.GenericKD")
print(event.describe())
print(isinstance(event, SecurityEvent))  # True

[2025-03-23 14:00] Severity: 8 | Malware: Trojan.GenericKD
True


### Polymorphism

When different classes share the same method name, you can call
that method on any of them without knowing the exact type. This
is **polymorphism**.

In [3]:
class IntrusionEvent(SecurityEvent):
    def __init__(self, timestamp, severity, source_ip):
        SecurityEvent.__init__(self, timestamp, severity)
        self.source_ip = source_ip

    def describe(self):
        base = SecurityEvent.describe(self)
        return base + " | Source: " + self.source_ip

# Polymorphism in action:
events = [
    MalwareEvent("2025-03-23 14:00", 8, "Trojan.GenericKD"),
    IntrusionEvent("2025-03-23 14:05", 9, "45.33.32.156"),
    SecurityEvent("2025-03-23 14:10", 3),
]

for e in events:
    print(e.describe())  # each calls its OWN version

[2025-03-23 14:00] Severity: 8 | Malware: Trojan.GenericKD
[2025-03-23 14:05] Severity: 9 | Source: 45.33.32.156
[2025-03-23 14:10] Severity: 3


### Composition ("has-a" relationship)

Instead of inheriting, an object can **contain** other objects.
For example, a `SecurityReport` *has* a list of events — it does
not *inherit* from an event.

In [4]:
class SecurityReport:
    def __init__(self, title):
        self.title = title
        self.events = []   # composition: has events

    def add_event(self, event):
        self.events.append(event)

    def get_high_severity(self, threshold):
        results = []
        for e in self.events:
            if e.severity >= threshold:
                results.append(e)
        return results

report = SecurityReport("Daily Summary")
report.add_event(MalwareEvent("2025-03-23 14:00", 8, "Trojan.GenericKD"))
report.add_event(SecurityEvent("2025-03-23 15:00", 2))

critical = report.get_high_severity(5)
for e in critical:
    print(e.describe())

[2025-03-23 14:00] Severity: 8 | Malware: Trojan.GenericKD


---
## Part 3: Guided Exercises (25-30 min)

### Exercise 1 — Inheritance: Scanner Hierarchy

Create a parent class `Scanner` with:
- `__init__(self, name)` — stores the scanner name.
- `scan(self, target)` — returns the string:
  `"<name> scanning <target>..."`

Create two child classes:
- `PortScanner(Scanner)` — overrides `scan` to return:
  `"<name> port-scanning <target> on ports 1-1024"`
- `VulnScanner(Scanner)` — overrides `scan` to return:
  `"<name> vulnerability-scanning <target> against CVE database"`

Create one of each and demonstrate polymorphism by looping
through a list of scanners and calling `scan("192.168.1.1")`.

In [ ]:
# YOUR CODE HERE

### Exercise 2 — Extending with `super()`

Create a class `NetworkDevice` with:
- `__init__(self, ip, hostname)` — stores both.
- `info(self)` — returns `"<hostname> (<ip>)"`

Create a child `Router(NetworkDevice)` with:
- `__init__` that also accepts `routing_table` (a list of strings)
  and calls the parent `__init__` using `super()` (or explicit parent call).
- `info(self)` — returns the parent info **plus**
  `" | Routes: <count>"` where count is the number of entries.

Create a Router and print its info.

In [ ]:
# YOUR CODE HERE

### Exercise 3 — Composition: Alert Dashboard

Create class `Alert` with:
- Attributes: `alert_type` (string), `message` (string), `severity` (int)
- Method `summary()` returning `"[<alert_type>] <message> (sev: <severity>)"`

Create class `Dashboard` with:
- A private list of Alert objects.
- `add_alert(alert)` — adds an alert.
- `filter_by_type(alert_type)` — returns a list of alerts matching
  that type.
- `highest_severity()` — returns the single alert with the highest
  severity value. If the list is empty, return `None`.

Test with at least 5 alerts of mixed types and severities.

In [ ]:
# YOUR CODE HERE

### Exercise 4 — Inheritance + Composition Together

Create a base class `Credential` with:
- `__init__(self, username)` and a method `authenticate(self, input_val)`.
  The base version always returns `False`.

Create two child classes:
- `PasswordCredential(Credential)` — `__init__` also takes `password`.
  `authenticate(input_val)` checks if `input_val == self._password`.
- `PinCredential(Credential)` — `__init__` also takes `pin` (a string
  of digits). `authenticate(input_val)` checks if `input_val == self._pin`.

Create a class `CredentialVault`:
- Stores a list of `Credential` objects.
- Method `add(credential)`.
- Method `authenticate_user(username, input_val)` — finds the credential
  with matching username and calls `authenticate`. Returns `True`/`False`.

Test with mixed credential types in the vault.

In [ ]:
# YOUR CODE HERE

---
## Part 4: Challenge Section (10 min)

### Challenge — Plugin-Style Log Analyzer

Design a mini framework where different analysis "plugins" can
be registered and run against a list of log entries.

1. Create a base class `AnalyzerPlugin` with:
   - `name` attribute (set in `__init__`)
   - Method `analyze(entries)` that returns an empty dictionary `{}`

2. Create child class `IPFrequencyAnalyzer(AnalyzerPlugin)`:
   - Overrides `analyze` to return a dictionary of IP -> count
     from the entries. Each entry is a dict with key `"ip"`.

3. Create child class `SeverityFilter(AnalyzerPlugin)`:
   - `__init__` also takes `min_severity` (int).
   - Overrides `analyze` to return a list of entries where
     `entry["severity"] >= self.min_severity`.

4. Create a class `LogAnalyzer`:
   - Stores a list of plugins and a list of log entries.
   - Method `add_plugin(plugin)`.
   - Method `add_entry(entry)` — entry is a dict.
   - Method `run_all()` — calls `analyze(self.entries)` on each
     plugin and prints the plugin name and its result.

Demonstrate with sample log data.

In [ ]:
# YOUR CODE HERE

---
## Submission
- Save and submit your completed notebook.
- Ensure all cells run cleanly.
- Submit via the course portal before the deadline.

**Excellent work — you now have the OOP toolkit to build
real-world cybersecurity applications!**